# TTS Engine Prototyping
It is a pain to restart the API Server since we need to unload/load the engine every time.

In [4]:
import signal
from contextlib import asynccontextmanager
from fastapi import FastAPI
import sys
import uvicorn
from queue import Queue
from RealtimeTTS import TextToAudioStream, CoquiEngine

In [8]:
audio_queue = Queue()
chunks_received = 0
engine = None
engine_stream = None
pending_feed = Queue()

def load_engine(model_name="xtts_v2", voice="voices/lance.wav"):
    global engine, engine_stream, pending_feed, audio_queue, chunks_received
    audio_queue = Queue()
    chunks_received = 0
    engine = CoquiEngine(model_name=model_name, voice=voice)
    engine_stream = TextToAudioStream(engine, on_audio_stream_stop=on_audio_stream_stop)
    pending_feed = Queue() 

def on_audio_stream_stop():
    print("Audio stream stopped")
    global audio_queue, pending_feed
    audio_queue.put(None)
    pending_feed = Queue()

In [27]:
load_engine()

In [10]:
import openai

In [11]:
client = openai.OpenAI(base_url="http://127.0.0.1:5000/v1", api_key="0608da5d28eb10cea2914f3de0f3ddba")

In [14]:
# ? Temporarily Stripped froom tts_api/runner.py
def openai_generator(gen_stream):
    """Generator for OpenAI streaming API.
    Yields sentences as they are completed."""
    payload = ""
    for chunk in gen_stream:
        if (content := chunk.choices[0].delta.content) is not None:
            # print(content, end="-")
            ends = ["?", ".", "!"]
            results = [end in content for end in ends]
            if any(results):
                payload += content
                # Get index of last found end in content
                last = max([payload.rindex(ends[i]) for i, x in enumerate(results) if x])
                feed, payload = payload[:last+1], payload[last+1:]
                # print("EOL")
                yield feed
            else:
                payload += content

    if payload.strip() != "":
        yield payload

In [36]:
import websockets

response = client.chat.completions.create(
    model="cognitivecomputations_dolphin-2.9-llama3-8b",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "tell me a short story (3 sentences)"
        }
    ],
    max_tokens=350,
    stream=True
)

# openai_gen = openai_generator(response)
# Establish a connection to the websocket server
websocket = await websockets.connect("ws://localhost:8000/ws")

async def gen():
    for chunk in response:
        print(chunk.choices[0].delta.content, end="")
        # Feed the chunk to the websocket
        await websocket.send(chunk.choices[0].delta.content)
        
        # Feed the chunk to the engine
        # yield chunk.choices[0].delta.content
    
    # Add a final END signal to the websocket
    await websocket.send("END")
    print("END")
    # Close the websocket connection
    await websocket.close()


await gen()

# engine_stream.stop()

# engine_stream.feed(gen()).play()

# 

Once upon a time, a young boy named Timmy loved to explore the forest near his home. One day, he discovered a hidden meadow filled with colorful wildflowers and lively butterflies. Overjoyed by this magical place, Timmy vowed to protect the meadow and share its beauty with the world.END


In [24]:
engine_stream.stop()